Netbook 2 : Nettoyage et transformation des données

Ce jalon couvre le nettoyage et les transformations colonne parcolonne.

Q11 — Valeurs nulles Pour chaque DataFrame, compter le nombre de valeurs nulles par colonne. Utiliser une boucle et
df.filter(col(c).isNull()).count().


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession \
    .builder \
    .appName("TradeCorp ETL") \
    .getOrCreate()


df_categories = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/home/jovyan/data/raw/categories.csv")
df_customers = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/home/jovyan/data/raw/customers.csv")
df_employees = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/home/jovyan/data/raw/employees.csv")
df_order_details = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/home/jovyan/data/raw/order_details.csv")
df_orders = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/home/jovyan/data/raw/orders.csv")
df_products = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/home/jovyan/data/raw/products.csv")
df_shippers = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/home/jovyan/data/raw/shippers.csv")
df_suppliers = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/home/jovyan/data/raw/suppliers.csv")

In [ ]:
# Valeurs nulles

dataframes = {
    "categories": df_categories,
    "customers": df_customers,
    "employees": df_employees,
    "order_details": df_order_details,
    "orders": df_orders,
    "products": df_products,
    "shippers": df_shippers,
    "suppliers": df_suppliers,
}


for nom, df in dataframes.items():
    print(f"--- {nom} ---")
    for c in df.columns:
        nb_nulls = df.filter(F.col(c).isNull()).count()
        print(f"{c}: {nb_nulls}")
    print()

--- categories ---
category_id: 0
category_name: 0
description: 0
picture: 8

--- customers ---
customer_id: 0
company_name: 0
contact_name: 0
contact_title: 0
address: 0
city: 0
region: 60
postal_code: 1
country: 0
phone: 0
fax: 22

--- employees ---
employee_id: 0
last_name: 0
first_name: 0
title: 0
title_of_courtesy: 0
birth_date: 0
hire_date: 0
address: 0
city: 0
region: 4
postal_code: 0
country: 0
home_phone: 0
extension: 0
photo: 9
notes: 0
reports_to: 1
photo_path: 0

--- order_details ---
order_id: 0
product_id: 0
unit_price: 0
quantity: 0
discount: 0

--- orders ---
order_id: 0
customer_id: 0
employee_id: 0
order_date: 0
required_date: 0
shipped_date: 21
ship_via: 0
freight: 0
ship_name: 0
ship_address: 0
ship_city: 0
ship_region: 507
ship_postal_code: 19
ship_country: 0

--- products ---
product_id: 0
product_name: 0
supplier_id: 0
category_id: 0
quantity_per_unit: 0
unit_price: 0
units_in_stock: 0
units_on_order: 0
reorder_level: 0
discontinued: 0

--- shippers ---
shipper_i

In [ ]:
# Supprimer les nulls

df_orders = df_orders.dropna(subset=["shipped_date"])



# remplacer les valeurs nulles de unit_price par la médiane.

median_price = df_products.agg(
    F.median("unit_price").alias("median_price")
).collect()[0]["median_price"]


df_products = df_products.fillna({"unit_price" : "median_price"})


In [ ]:
# Cast des types

from pyspark.sql.types import DateType , DoubleType ,IntegerType


df_orders = (
    df_orders
    .withColumn("order_date", df_orders.order_date.cast(DateType()))
    .withColumn("required_date", df_orders.required_date.cast(DateType()))
    .withColumn("shipped_date", df_orders.shipped_date.cast(DateType()))
)

# unit_price

df_order_details = (df_order_details.withColumn("unit_price",df_order_details.unit_price.cast(DoubleType())))

df_order_details = (df_order_details.withColumn("unit_price",df_order_details.quantity.cast(IntegerType())))


In [ ]:
# Nettoyage des chaînes

from pyspark.sql.types import StringType

for c , t in df_customers.dtypes:
    if t == "string":
        df_customers = df_customers.withColumn( c ,F.trim(F.col(c)))



# initcap().
df_customers = df_customers.withColumn("contact_name" ,F.initcap("contact_name"))


#  upper().
df_customers = df_customers.withColumn("country" ,F.upper("country"))


In [ ]:
# Renommer les colonnes

df_order_details = df_order_details.withColumnRenamed("unit_price" , "prix_unitaire" )

df_order_details = df_order_details.withColumnRenamed("quantity" , "quantite" )


df_orders = df_orders.withColumnRenamed("ship_via" , "shipper_id")



In [ ]:
# Colonnes calculées

df_order_details = df_order_details.withColumn("sous_total" ,F.round(F.col("prix_unitaire") * F.col("quantite") * (1 - F.col('discount')),2))



In [ ]:
# Colonnes conditionnelles
df_products = df_products.withColumn("en_stock" , F.when(F.col("units_in_stock") > 0, True).otherwise(False))

df_orders = df_orders.withColumn("is_shipped", F.when(F.col("shipped_date").isNotNull(), True).otherwise(False))
                    

In [ ]:
# Doublons

df_customers.select("customer_id").distinct().count()
print(f" total_ligne_distinct {df_customers.count()}")

 total_ligne_distinct 91


In [ ]:
print(f" total_lignes {df_customers.select("customer_id").count()}")

 total_lignes 91


In [ ]:
# Supprimer les doublons 
df_customers = df_customers.dropDuplicates(subset=["customer_id"])

In [ ]:
# Filtrage
# que les commandes de 1997. 

df_orders = df_orders.filter(F.col("order_date").between("1997-01-01" , "1997-12-31"))


# que les produits en stock

df_products = df_products.filter((F.col("units_in_stock") > 0 )& (F.col("discontinued") == 0))


In [ ]:
# Sélection de colonnes

df_employees = (
    df_employees
    .select(
        F.col("employee_id"),
        F.col("first_name"), 
        F.col("last_name"), 
        F.col("title"), 
        F.col("hire_date"), 
        F.col("city"),
        F.col("country")
))

In [ ]:
# création de collonne (concat)

df_employees = df_employees.withColumn("full_name", F.concat(F.col("first_name") , F.lit(" "), F.col("last_name")))

In [ ]:
# Enregistrer les dataframe en format parquet

dataframesclean = {
    "categories": df_categories,
    "customers": df_customers,
    "employees": df_employees,
    "order_details": df_order_details,
    "orders": df_orders,
    "products": df_products,
    "shippers": df_shippers,
    "suppliers": df_suppliers,
}

PATH = "/home/jovyan/data/tmp"

for name, df in dataframesclean.items():
    output_path = f"{PATH}/{name}"
    
    df.write.mode("overwrite").parquet(output_path)
    print(f"--- {name} sauvegardé dans {output_path} ---")

--- categories sauvegardé dans /home/jovyan/data/tmp/categories ---
--- customers sauvegardé dans /home/jovyan/data/tmp/customers ---
--- employees sauvegardé dans /home/jovyan/data/tmp/employees ---
--- order_details sauvegardé dans /home/jovyan/data/tmp/order_details ---
--- orders sauvegardé dans /home/jovyan/data/tmp/orders ---
--- products sauvegardé dans /home/jovyan/data/tmp/products ---
--- shippers sauvegardé dans /home/jovyan/data/tmp/shippers ---
--- suppliers sauvegardé dans /home/jovyan/data/tmp/suppliers ---
